# Assessment 2: Machine learning and real-time streaming

## Stefan Garevski (33759839)

### Dataset reuse and train/stream split

In [1]:
#initial imports
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col,
    when,
    hour,
    to_timestamp,
    countDistinct,
    sum,
    round,
    substring,
    row_number,
    spark_partition_id,
    min,
    max,
    avg,
    count
)
from pyspark.sql.window import Window

#initiate spark session
spark = SparkSession.builder \
    .appName("Victorian Road Crash Analysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 20:41:16 WARN Utils: Your hostname, Stefans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.118 instead (on interface en0)
26/09/22 20:41:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/22 20:41:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#define schema of variables in dataset 
schema = StructType([
    StructField("ACCIDENT_NO", StringType(), True),
    StructField("ACCIDENT_DATE", StringType(), True),
    StructField("ACCIDENT_TIME", StringType(), True),
    StructField("ACCIDENT_TYPE", StringType(), True),
    StructField("DAY_OF_WEEK", StringType(), True),
    StructField("DCA_CODE", StringType(), True),
    StructField("DCA_CODE_DESCRIPTION", StringType(), True),
    StructField("LIGHT_CONDITION", StringType(), True),
    StructField("POLICE_ATTEND", StringType(), True),
    StructField("ROAD_GEOMETRY", StringType(), True),
    StructField("SEVERITY", StringType(), True),
    StructField("SPEED_ZONE", StringType(), True),
    StructField("RUN_OFFROAD", StringType(), True),
    StructField("ROAD_NAME", StringType(), True),
    StructField("ROAD_TYPE", StringType(), True),
    StructField("ROAD_ROUTE_1", StringType(), True),
    StructField("LGA_NAME", StringType(), True),
    StructField("DTP_REGION", StringType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
    StructField("VICGRID_X", DoubleType(), True),
    StructField("VICGRID_Y", DoubleType(), True),
    StructField("TOTAL_PERSONS", IntegerType(), True),
    StructField("INJ_OR_FATAL", IntegerType(), True),
    StructField("FATALITY", IntegerType(), True),
    StructField("SERIOUSINJURY", IntegerType(), True),
    StructField("OTHERINJURY", IntegerType(), True),
    StructField("NONINJURED", IntegerType(), True),
    StructField("MALES", IntegerType(), True),
    StructField("FEMALES", IntegerType(), True),
    StructField("BICYCLIST", IntegerType(), True),
    StructField("PASSENGER", IntegerType(), True),
    StructField("DRIVER", IntegerType(), True),
    StructField("PEDESTRIAN", IntegerType(), True),
    StructField("PILLION", IntegerType(), True),
    StructField("MOTORCYCLIST", IntegerType(), True),
    StructField("UNKNOWN", IntegerType(), True),
    StructField("PED_CYCLIST_5_12", IntegerType(), True),
    StructField("PED_CYCLIST_13_18", IntegerType(), True),
    StructField("OLD_PED_65_AND_OVER", IntegerType(), True),
    StructField("OLD_DRIVER_75_AND_OVER", IntegerType(), True),
    StructField("YOUNG_DRIVER_18_25", IntegerType(), True),
    StructField("NO_OF_VEHICLES", IntegerType(), True),
    StructField("HEAVYVEHICLE", IntegerType(), True),
    StructField("PASSENGERVEHICLE", IntegerType(), True),
    StructField("MOTORCYCLE", IntegerType(), True),
    StructField("PT_VEHICLE", IntegerType(), True),
    StructField("DEG_URBAN_NAME", StringType(), True),
    StructField("SRNS", StringType(), True),
    StructField("RMA", StringType(), True),
    StructField("DIVIDED", StringType(), True),
    StructField("STAT_DIV_NAME", StringType(), True)
])

In [3]:
#load in csv data
file_path = "../Assessment 1/vic_road_crash_data.csv"

crashes = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv(file_path)

In [4]:
# create split
train_df, stream_df = crashes.randomSplit([0.7, 0.3], seed=42)
total = crashes.count()

# check percentage split
print("Training:", train_df.count(), f"({train_df.count()/total:.2%})")
print("Streaming:", stream_df.count(), f"({stream_df.count()/total:.2%})")

# save as parquet
stream_df.write.mode("overwrite").parquet("data/stream_data.parquet")

26/09/22 20:41:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Training: 140622 (70.19%)
Streaming: 59730 (29.81%)


26/09/22 20:41:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/22 20:41:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/22 20:41:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/22 20:41:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/22 20:41:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [17]:
stream_df.select("STAT_DIV_NAME").distinct().count()

3

# Part A: The ML pipeline (45%)

## 1. ML task selection

For Part A, I have selected Classification as the machine learning task. The rationale is that the VicRoads crash dataset contains an appropriate categorical target variable called `SEVERITY`, which represents the severity category of each recorded road crash.

The dataset contains a mix of categorical and numerical variables that can be used as features in the model. Categorical variables include `ACCIDENT_TYPE`, `DAY_OF_WEEK`, `LIGHT_CONDITION`, `ROAD_GEOMETRY`, `ROAD_TYPE`, `SPEED_ZONE`, `RUN_OFFROAD`, and `LGA_NAME`. Numerical variables include `TOTAL_PERSONS`, `INJ_OR_FATAL`, `FATALITY`, `SERIOUSINJURY`, `OTHERINJURY`, `NO_OF_VEHICLES`, etc. These variables describe the crash, road environment, location, vehicles, and people involved. Therefore these variables can provide meaningful information for predicting the crash severity.

Classification is the most suitable in this instance because the objective is to predict which severity category a crash belongs to rather than predict a continuous numerical quantity. The `SEVERITY` column provides an existing categorical label that can be used, so there is no requirement to set any type of threshold for this activity.

Something like a regression is less suitable because most of the numerical variables generally describe counts or measurements associated with a crash rather than a prediction whose output would be a continuous number. Also, the `SEVERITY` variable is categorical rather than continuous.

Clustering is also not really suitable because it is primarily intended for situations where there may be no suitable target variable and the goal is to discover previously unknown groups or patterns in the data. In this instance, classification provides a more direct way to use the Victorian crash dataset.

Therefore, I will be using classification with `SEVERITY` as the target variable and relevant crash, road, vehicle, location, and environmental attributes as predictive features.

## 2. Feature engineering

In [29]:
# bring in packages for feature engineering purposes
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

# define potential categorical variables
pot_categorical_cols = [
    "ACCIDENT_TYPE",
    "DAY_OF_WEEK",
    "LIGHT_CONDITION",
    "POLICE_ATTEND",
    "ROAD_GEOMETRY",
    "SPEED_ZONE",
    "RUN_OFFROAD",
    "ROAD_NAME",
    "ROAD_TYPE",
    "LGA_NAME",
    "DTP_REGION",
    "DEG_URBAN_NAME",
    "SRNS",
    "RMA",
    "DIVIDED",
    "STAT_DIV_NAME",
    "DCA_CODE",
    "DCA_CODE_DESCRIPTION"
]

#define potential numerical variables
pot_numeric_cols = [
    "LATITUDE",
    "LONGITUDE",
    "TOTAL_PERSONS",
    "NO_OF_VEHICLES",
    "MALES",
    "FEMALES",
    "BICYCLIST",
    "PASSENGER",
    "DRIVER",
    "PEDESTRIAN",
    "PILLION",
    "MOTORCYCLIST",
    "UNKNOWN",
    "PED_CYCLIST_5_12",
    "PED_CYCLIST_13_18",
    "OLD_PED_65_AND_OVER",
    "OLD_DRIVER_75_AND_OVER",
    "HEAVYVEHICLE",
    "YOUNG_DRIVER_18_25",
    "PASSENGERVEHICLE",
    "MOTORCYCLE",
    "PT_VEHICLE"
]

In [30]:
#check categorical feature cardinality
from pyspark.sql import functions as F

for col in pot_categorical_cols:
    stats = crashes.select(
        F.countDistinct(col).alias("distinct"),
        F.count("*").alias("total")
    ).collect()[0]

    print(
        f"{col:25} "
        f"distinct={stats['distinct']:5} "
        f"total={stats['total']}"
    )

ACCIDENT_TYPE             distinct=    9 total=200352
DAY_OF_WEEK               distinct=    7 total=200352
LIGHT_CONDITION           distinct=    7 total=200352
POLICE_ATTEND             distinct=    3 total=200352
ROAD_GEOMETRY             distinct=    9 total=200352
SPEED_ZONE                distinct=   13 total=200352
RUN_OFFROAD               distinct=    2 total=200352
ROAD_NAME                 distinct=14558 total=200352
ROAD_TYPE                 distinct=   80 total=200352
LGA_NAME                  distinct=   87 total=200352
DTP_REGION                distinct=    8 total=200352
DEG_URBAN_NAME            distinct=    7 total=200352
SRNS                      distinct=    4 total=200352
RMA                       distinct=    6 total=200352
DIVIDED                   distinct=    2 total=200352
STAT_DIV_NAME             distinct=    2 total=200352
DCA_CODE                  distinct=   81 total=200352
DCA_CODE_DESCRIPTION      distinct=   81 total=200352


In [31]:
#check numerical variable variance
for col in pot_numeric_cols:
    stats = crashes.select(
        F.countDistinct(col).alias("distinct"),
        F.min(col).alias("min"),
        F.max(col).alias("max"),
        F.avg(col).alias("mean")
    ).collect()[0]

    print(
        f"{col:25} "
        f"distinct={stats['distinct']:5} "
        f"min={stats['min']} "
        f"max={stats['max']} "
        f"mean={stats['mean']:.2f}"
    )

LATITUDE                  distinct=106932 min=-39.03083 max=-34.115696 mean=-37.72
LONGITUDE                 distinct=79276 min=140.96648 max=149.75746 mean=144.97
TOTAL_PERSONS             distinct=   45 min=1 max=97 mean=2.36
NO_OF_VEHICLES            distinct=   16 min=1 max=21 mean=1.82
MALES                     distinct=   26 min=0 max=46 mean=1.32
FEMALES                   distinct=   30 min=0 max=51 mean=0.94
BICYCLIST                 distinct=    9 min=0 max=8 mean=0.10
PASSENGER                 distinct=   42 min=0 max=95 mean=0.48
DRIVER                    distinct=   16 min=0 max=21 mean=1.49
PEDESTRIAN                distinct=   10 min=0 max=11 mean=0.09
PILLION                   distinct=    3 min=0 max=2 mean=0.01
MOTORCYCLIST              distinct=    9 min=0 max=9 mean=0.14
UNKNOWN                   distinct=   24 min=0 max=52 mean=0.05
PED_CYCLIST_5_12          distinct=    6 min=0 max=8 mean=0.01
PED_CYCLIST_13_18         distinct=    5 min=0 max=5 mean=0.02
OLD_PED_6

In [ ]:
# define categorical variables
categorical_cols = [
    "ACCIDENT_TYPE",
    "DAY_OF_WEEK",
    "LIGHT_CONDITION",
    "POLICE_ATTEND",
    "ROAD_GEOMETRY",
    "SPEED_ZONE",
    "RUN_OFFROAD",
    "DTP_REGION",
    "DEG_URBAN_NAME",
    "DIVIDED",
    "STAT_DIV_NAME"
]

#define numerical variables
numeric_cols = [
    "TOTAL_PERSONS",
    "NO_OF_VEHICLES",
    "MALES",
    "FEMALES",
    "BICYCLIST",
    "PASSENGER",
    "DRIVER",
    "PEDESTRIAN",
    "PILLION",
    "MOTORCYCLIST",
    "UNKNOWN",
    "PED_CYCLIST_5_12",
    "PED_CYCLIST_13_18",
    "OLD_PED_65_AND_OVER",
    "OLD_DRIVER_75_AND_OVER",
    "HEAVYVEHICLE",
    "YOUNG_DRIVER_18_25",
    "PASSENGERVEHICLE",
    "MOTORCYCLE",
    "PT_VEHICLE"
]

In [26]:
# set the target variable
label_indexer = StringIndexer(
    inputCol="SEVERITY",
    outputCol="label",
    handleInvalid="keep"
)

# convert categorical variables to numerical indices
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_index",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

# one hot encode for categorical variables
encoder = OneHotEncoder(
    inputCols=[col + "_index" for col in categorical_cols],
    outputCols=[col + "_encoded" for col in categorical_cols]
)

# combine categorical and numerical variables into single feature vector
assembler = VectorAssembler(
    inputCols=(
        [col + "_encoded" for col in categorical_cols]
        + numeric_cols
    ),
    outputCol="features",
    handleInvalid="keep"
)

# create the preprocessing pipeline
preprocessing_pipeline = Pipeline(
    stages=indexers + [encoder, label_indexer, assembler]
)

In [33]:
# complete train/eval split
train_data, test_data = train_df.randomSplit([0.8, 0.2], seed=42)

In [35]:
# fit preprocessing pipeline using training data only
preprocessor_model = preprocessing_pipeline.fit(train_data)

# transform the training data
train_prepared = preprocessor_model.transform(train_data)

# transform the unseen test data using the fitted preprocessing pipeline
test_prepared = preprocessor_model.transform(test_data)

### Documentation

Feature selection was performed with three objectives; prevent target leakage, retain variables with meaningful predictive information, and remove features demonstrating high-cardinality or low information.

`SEVERITY` is the classification target and is therefore excluded from the list of variables for modelling. It is transformed separately using `StringIndexer` to produce the label column required by spark.

The variables `FATALITY`, `SERIOUSINJURY`, `OTHERINJURY`, `INJ_OR_FATAL`, and `NONINJURED` are excluded because they describe injury and fatality outcomes resulting from the crash. These variables are closely related to the crash severity outcome and could allow the model to infer the target using post-crash information rather than learning from the circumstances associated with the crash. Excluding these variables reduces the risk of target leakage.

`ACCIDENT_NO` is excluded because it is an identifier rather than providing any meaningful information. `DCA_CODE` and `DCA_CODE_DESCRIPTION` are also excluded because they provide detailed crash classifications that may encode information closely associated with the crash outcome.

The selected categorical describe crash type, temporal conditions, lighting, road characteristics, speed environment, road configuration, and geographic information. They are retained because these characteristics may provide useful predictive information.

Several categorical variables were not included in the final feature vector. For example, `ROAD_NAME` and `LGA_NAME` were excluded because they can contain many distinct categories. One-hot encoding very high-cardinality variables can substantially increase the dimensionality of the feature vector, increase computational cost, and potentially encourage the model to learn location-specific patterns rather than general relationships. The broader geographic variables `DTP_REGION`, `DEG_URBAN_NAME`, and `STAT_DIV_NAME` were retained to provide geographic context at a more manageable level of granularity.

The selected numerical variables describe the number and composition of people and vehicles involved in each crash. These include `TOTAL_PERSONS`, `NO_OF_VEHICLES`, demographic counts, road-user counts, and vehicle-type counts.

`LATITUDE` and `LONGITUDE` were excluded from the final feature vector. Although they may contain predictive geographic information, excluding precise coordinates reduces the dependence of the model on exact location and keeps the feature set focused on crash, road, environmental, demographic, and vehicle characteristics.

Numerical features were also inspected for variation so that variables containing no meaningful variation could be identified and excluded. Variables with effectively zero variance would provide no discriminatory information to a classifier and would unnecessarily increase the feature vector.

The categorical variables are stored as strings and therefore cannot be directly supplied to the selected Spark ML classifier. `StringIndexer` converts each categorical value into a numerical category index. `OneHotEncoder` then represents these categories as binary indicator variables.

One-hot encoding is important because the numerical indices produced are arbitrary category identifiers and do not imply an ordinal relationship between categories.

The `handleInvalid="keep"` option is used so that previously unseen or invalid categorical values can be represented by an additional category rather than causing the transformation to fail. This is particularly important for the held-out streaming data, where a category may occur that was not present in the training subset.

`VectorAssembler` combines the one-hot encoded categorical variables and selected numerical variables into a single `features` vector. This is the input into spark ML classification algorithm.

The transformation pipeline is done in the following order:

1. `StringIndexer` transforms the categorical predictor variables
2. `OneHotEncoder` converts the indexed categorical variables into binary feature vectors
3. `StringIndexer` converts the `SEVERITY` target into the numerical `label` required by spark 
4. `VectorAssembler` combines the encoded categorical variables and numerical variables into the final `features` vector

## 3. Model comparison

## 4. Evaluation requirements

## 5. Model persistence

# Part B: Streaming, Kafka and reflection

## 1. Streaming data simulation

## 2. Kafka producer

## 3. Spark structured streaming consumer

## 4. Windowed aggregation

## 5. Reflection section